# Explore dataset structure

In [ ]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)   # go one level up
print(os.getcwd())         

from xflow import TransformRegistry as T
from config_utils import load_config, detect_machine
from utils import *

experiment_name = "CAE_validate_clear"  
machine = detect_machine() 
config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

dataset = "DMD_only" # "DMD_only" or "Chromox_only"

# if win, windows test in machine then auto switch to windows path, otherwise use mac path
if any(win in machine for win in ["win", "windows"]): # windows
    dirs = {
        "DMD_only": {
            "dataset_db_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/CLEAR25_DMD/db/dataset_meta.db",
            "dataset_extracted_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/CLEAR25_DMD/",
        },
        "Chromox_only": {
            "dataset_db_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/CLEAR25_Chromox/db/dataset_meta.db",
            "dataset_extracted_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/CLEAR25_Chromox/",
        }
    }

elif any(mac in machine for mac in ["mac", "darwin"]): # Mac
    dirs = {
        "DMD_only": {
            "dataset_db_dir": "",
        },
        "Chromox_only": {
            "dataset_db_dir": "",
        }
    }

else:
    raise ValueError(f"Unsupported machine: {machine}")

# Load dataset

In [ ]:
# pip install xflow-py
from xflow import SqlProvider, PyTorchPipeline
from xflow.data import build_transforms_from_config
from xflow.extensions.physics.pipeline import CachedBasisPipeline, IndexCombinator
from xflow.extensions.physics import pattern_gen

train_provider = SqlProvider(
    sources={"connection": dirs[dataset]["dataset_db_dir"], "sql": config["sql"]["train"]}, output_config={'list': "image_path"}
)
eval_provider = SqlProvider(
    sources={"connection": dirs[dataset]["dataset_db_dir"], "sql": config["sql"]["eval"]}, output_config={'list': "image_path"}
)
# create data pipelines for ML training and evaluation
config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": dirs[dataset]["dataset_extracted_dir"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

# ========== SGM simulation pattern generator ==========
canvas = pattern_gen.DynamicPatterns(*config["simulation"]["canvas_size"])
canvas.set_postprocess_fns(build_transforms_from_config(config["simulation"]["process_functions"]))
canvas._distributions = [pattern_gen.StaticGaussianDistribution(canvas) for _ in range(config["simulation"]["total_Guassian_num"])]
canvas.set_threshold(config["simulation"]["minimum_pixel_threshold"])
stream = canvas.pattern_stream(std_1=config["simulation"]["std_1"], std_2=config["simulation"]["std_2"],
                               max_intensity=config["simulation"]["max_intensity"], fade_rate=config["simulation"]["fade_rate"], 
                               distribution=config["simulation"]["distribution"]) 

# ======== random combinator using index + SGM ========
combinator = IndexCombinator(
    pattern_provider=stream,
    transforms= build_transforms_from_config(config["combinator"]["transforms"]["torch"]),
)

val_provider, test_provider = eval_provider.split(config["data"]["val_test_split"])
train_dataset = CachedBasisPipeline(
    train_provider, 
    combinator=combinator, 
    transforms=transforms, 
    num_samples=config["data"]["total_train_samples"], 
    seed=config["seed"],
    eager=True
).to_framework_dataset(framework=config["framework"], dataset_ops=config["data"]["dataset_ops"])
val_dataset = PyTorchPipeline(
    val_provider, 
    transforms[:-1]
).to_memory_dataset(config["data"]["dataset_ops"])   # testset data do not need thresholding since it is to remove stacking noise?
test_dataset = PyTorchPipeline(
    test_provider,
    transforms[:-1]
).to_memory_dataset(config["data"]["dataset_ops"])

print("Samples: ",len(train_provider),len(val_provider),len(test_provider))
print("Batch: ",len(train_dataset),len(val_dataset),len(test_dataset))

# merge to single provider -> pipeline
dmd_provider = train_provider.merge(eval_provider)
dmd_pipeline = PyTorchPipeline(
    dmd_provider,
    transforms[:-1]
).to_numpy()

# Latent space distribution

In [ ]:
from xflow.utils import plot_image

plot_image(dmd_pipeline[1][0])

In [ ]:
from xflow.utils.visualization import DimReducer, plot_embedding

reducer = DimReducer(method="pca", n_components=3, random_state=42)

X = dmd_pipeline[0].reshape(dmd_pipeline[0].shape[0], -1)  # change shape to (n_samples, n_features)
coords = reducer.fit_transform(X)
plot_embedding(coords,  title="PCA projection - MMF output speckle")   

X = dmd_pipeline[1].reshape(dmd_pipeline[1].shape[0], -1)  # change shape to (n_samples, n_features)
coords = reducer.fit_transform(X)
plot_embedding(coords,  title="PCA projection - Original image")   

In [ ]:
26**5